# Agent (智能体) 学习

## 什么是 Agent?

Agent = 感知环境 + 自主决策 + 执行动作 + 达成目标

```
传统AI:  输入 -> 模型 -> 输出 (单次推理)
Agent:   感知 -> 思考 -> 行动 -> 观察 -> 思考 -> 行动 -> ... (循环)
```

核心区别: Agent 可以**多步推理**、**调用工具**、**与环境交互**。

## 本课内容
1. Agent 基础框架
2. ReAct: 推理+行动
3. 工具调用 (Tool Use)
4. RAG: 检索增强生成
5. 规划与记忆
6. Multi-Agent 系统
7. Agent 开发框架

In [ ]:
import numpy as np
import json
import re
from typing import List, Dict, Optional, Callable
from dataclasses import dataclass, field
print("Agent 学习环境准备完成")

## 1. Agent 基础框架

### Agent 的核心组件

```
Agent
|-- 感知 (Perception): 接收环境信息
|-- 大脑 (Brain):      LLM / 规则引擎 / 深度学习模型
|-- 记忆 (Memory):     短期记忆 + 长期记忆
|-- 规划 (Planning):   任务分解 + 步骤规划
|-- 工具 (Tools):      可调用的外部能力
|-- 行动 (Action):     执行决策
+-- 反馈 (Feedback):   观察行动结果
```

### Agent 循环

```
while not done:
    observation = perceive(environment)    # 感知
    thought = think(observation, memory)    # 思考
    action = decide(thought, tools)         # 决策
    result = execute(action)                # 执行
    memory.update(observation, action, result)  # 记忆
    done = check_goal(result)               # 检查目标
```

In [ ]:
# 基础Agent框架
@dataclass
class Message:
    role: str  # system, user, assistant, tool
    content: str
    name: Optional[str] = None

class BaseAgent:
    def __init__(self, name: str = "Agent", max_steps: int = 10):
        self.name = name
        self.max_steps = max_steps
        self.memory: List[Message] = []
        self.tools: Dict[str, Callable] = {}
    
    def add_tool(self, name: str, func: Callable, description: str = ""):
        self.tools[name] = func
    
    def observe(self, message: Message):
        self.memory.append(message)
    
    def think(self) -> str:
        raise NotImplementedError
    
    def act(self, thought: str) -> str:
        raise NotImplementedError
    
    def run(self, task: str) -> str:
        self.observe(Message(role="user", content=task))
        for step in range(self.max_steps):
            thought = self.think()
            result = self.act(thought)
            self.observe(Message(role="assistant", content=f"Step {step+1}: {thought}"))
            self.observe(Message(role="tool", content=result))
            if self._is_done(result):
                return result
        return "Max steps reached"
    
    def _is_done(self, result: str) -> bool:
        return "FINAL ANSWER:" in result

print("BaseAgent 框架定义完成")

---
## 2. ReAct: 推理+行动

ReAct (Reasoning + Acting) 是最经典的Agent框架:

```
Thought: 我需要查找设备3的射频指纹信息
Action:  lookup_device(device_id=3)
Observation: 设备3, WiFi, CFO=500Hz, IQ_amp=1.05
Thought: 现在我有了设备3的信息, 可以进行认证
Action:  authenticate(iq_data=..., device_id=3)
Observation: 认证成功, 置信度0.95
Thought: 认证完成, 我可以给出最终答案
Final Answer: 设备3认证成功, 置信度95%
```

关键: **思考过程可解释**, 每一步都有明确的推理。

In [ ]:
# ReAct Agent 实现
class ReActAgent(BaseAgent):
    def __init__(self, name: str = "ReActAgent", max_steps: int = 10):
        super().__init__(name, max_steps)
        self.device_db = {
            0: {"name": "Sensor-A", "type": "wifi", "cfo": 200, "location": "Lab-1"},
            1: {"name": "Sensor-B", "type": "ble", "cfo": -150, "location": "Lab-2"},
            2: {"name": "Sensor-C", "type": "lora", "cfo": 350, "location": "Lab-3"},
            3: {"name": "Sensor-D", "type": "wifi", "cfo": -80, "location": "Lab-1"},
            4: {"name": "Sensor-E", "type": "zigbee", "cfo": 420, "location": "Lab-4"},
        }
        self._register_tools()
    
    def _register_tools(self):
        self.add_tool("lookup_device", self._lookup_device, "查询设备信息")
        self.add_tool("authenticate", self._authenticate, "认证设备")
        self.add_tool("check_snr", self._check_snr, "检查信噪比")
        self.add_tool("list_devices", self._list_devices, "列出所有设备")
    
    def _lookup_device(self, device_id: int) -> str:
        if device_id in self.device_db:
            return json.dumps(self.device_db[device_id])
        return f"Device {device_id} not found"
    
    def _authenticate(self, device_id: int, confidence: float = 0.0) -> str:
        if device_id not in self.device_db:
            return f"Device {device_id} not in database - REJECTED"
        if confidence >= 0.7:
            return f"Device {device_id} authenticated (confidence={confidence:.2f})"
        return f"Device {device_id} rejected (confidence={confidence:.2f} < 0.7)"
    
    def _check_snr(self, device_id: int) -> str:
        snr = np.random.uniform(5, 30)
        return f"Device {device_id} SNR: {snr:.1f} dB"
    
    def _list_devices(self) -> str:
        return json.dumps({k: v["name"] for k, v in self.device_db.items()})
    
    def think(self) -> str:
        return self._rule_based_think()
    
    def _rule_based_think(self) -> str:
        last_user_msg = ""
        for m in reversed(self.memory):
            if m.role == "user":
                last_user_msg = m.content
                break
        return last_user_msg
    
    def act(self, thought: str) -> str:
        return self._parse_and_execute(thought)
    
    def _parse_and_execute(self, text: str) -> str:
        # 简单的指令解析
        if "list" in text.lower() and "device" in text.lower():
            return self.tools["list_devices"]()
        
        dev_id_match = re.search(r"device[_ ]?(\d+)", text, re.IGNORECASE)
        dev_id = int(dev_id_match.group(1)) if dev_id_match else None
        
        if dev_id is None:
            return "Please specify a device ID"
        
        if "authenticate" in text.lower() or "auth" in text.lower():
            conf = np.random.uniform(0.6, 0.99)
            return self.tools["authenticate"](dev_id, conf)
        elif "snr" in text.lower() or "signal" in text.lower():
            return self.tools["check_snr"](dev_id)
        elif "lookup" in text.lower() or "find" in text.lower() or "info" in text.lower():
            return self.tools["lookup_device"](dev_id)
        else:
            return self.tools["lookup_device"](dev_id)

agent = ReActAgent()
print("ReActAgent 创建完成")

In [ ]:
# 运行Agent
print("=== Agent 交互演示 ===")
print()

# 任务1: 查询设备信息
agent1 = ReActAgent()
result = agent1.run("lookup device 2")
print("任务: 查询设备2")
for m in agent1.memory:
    print(f"  [{m.role}] {m.content[:80]}")
print()

# 任务2: 认证设备
agent2 = ReActAgent()
result = agent2.run("authenticate device 3")
print("任务: 认证设备3")
for m in agent2.memory:
    print(f"  [{m.role}] {m.content[:80]}")
print()

# 任务3: 列出设备
agent3 = ReActAgent()
result = agent3.run("list devices")
print(f"任务: 列出设备 -> {result}")

---
## 3. 工具调用 (Tool Use)

Agent 的能力来自工具。定义好工具, Agent 就能完成各种任务。

```
工具定义:
  name:       工具名称
  description: 功能描述 (LLM根据描述选择工具)
  parameters:  参数定义 (类型、约束)
  function:    实际执行函数
```

In [ ]:
# 工具定义框架
@dataclass
class Tool:
    name: str
    description: str
    parameters: Dict
    function: Callable
    
    def run(self, **kwargs):
        return self.function(**kwargs)
    
    def get_schema(self) -> Dict:
        return {
            "name": self.name,
            "description": self.description,
            "parameters": self.parameters,
        }

# 定义RFFI相关工具
def rf_scan(frequency_mhz: float, bandwidth_mhz: float = 20.0) -> str:
    n_devices = np.random.randint(1, 6)
    devices = [f"Device-{i}" for i in range(n_devices)]
    return json.dumps({"frequency": frequency_mhz, "bandwidth": bandwidth_mhz,
                       "devices_found": devices, "count": n_devices})

def rf_fingerprint(iq_data_hex: str) -> str:
    np.random.seed(hash(iq_data_hex) % 2**31)
    device_id = np.random.randint(0, 5)
    confidence = np.random.uniform(0.7, 0.99)
    return json.dumps({"device_id": device_id, "confidence": round(confidence, 4),
                       "method": "1D-CNN"})

def rf_jamming_detect(power_dbm: float, threshold_dbm: float = -50.0) -> str:
    is_jammed = power_dbm > threshold_dbm
    return json.dumps({"power_dbm": power_dbm, "threshold": threshold_dbm,
                       "is_jammed": is_jammed, "severity": "high" if is_jammed else "none"})

tools = [
    Tool(name="rf_scan", description="扫描指定频率的射频信号, 发现附近设备",
         parameters={"frequency_mhz": {"type": "float", "required": True},
                     "bandwidth_mhz": {"type": "float", "default": 20.0}},
         function=rf_scan),
    Tool(name="rf_fingerprint", description="分析IQ数据提取射频指纹, 识别设备",
         parameters={"iq_data_hex": {"type": "string", "required": True}},
         function=rf_fingerprint),
    Tool(name="rf_jamming_detect", description="检测是否存在射频干扰/压制",
         parameters={"power_dbm": {"type": "float", "required": True},
                     "threshold_dbm": {"type": "float", "default": -50.0}},
         function=rf_jamming_detect),
]

print("工具定义:")
for t in tools:
    print(f"  {t.name}: {t.description}")

# 测试工具
print(f"\n扫描测试: {tools[0].run(frequency_mhz=2400)}")
print(f"指纹测试: {tools[1].run(iq_data_hex='a1b2c3')}")
print(f"干扰测试: {tools[2].run(power_dbm=-30)}")

In [ ]:
# Function Calling 格式 (OpenAI风格)
print("=== Function Calling 格式 ===")
print()
print("LLM 输出:")
print(json.dumps({
    "role": "assistant",
    "content": None,
    "tool_calls": [{
        "id": "call_001",
        "type": "function",
        "function": {
            "name": "rf_fingerprint",
            "arguments": "{\"iq_data_hex\": \"a1b2c3d4\"}"
        }
    }]
}, indent=2, ensure_ascii=False))
print()
print("工具返回:")
print(json.dumps({
    "role": "tool",
    "tool_call_id": "call_001",
    "content": "{\"device_id\": 2, \"confidence\": 0.95}"
}, indent=2, ensure_ascii=False))
print()
print("流程: 用户提问 -> LLM选择工具 -> 执行工具 -> 结果返回LLM -> LLM生成回答")

---
## 4. RAG: 检索增强生成

RAG 让 Agent 能访问外部知识库, 解决 LLM 知识过时/不足的问题。

```
用户问题 -> 检索相关文档 -> 拼接上下文 -> LLM生成回答

关键步骤:
1. 文档分块 (Chunking)
2. 向量化 (Embedding)
3. 向量检索 (Similarity Search)
4. 上下文注入 (Context Injection)
```

In [ ]:
# 简单RAG实现
class SimpleRAG:
    def __init__(self):
        self.documents = []
        self.embeddings = []
    
    def add_document(self, doc_id: str, text: str, metadata: Dict = None):
        self.documents.append({"id": doc_id, "text": text, "metadata": metadata or {}})
        embedding = self._simple_embed(text)
        self.embeddings.append(embedding)
    
    def _simple_embed(self, text: str) -> np.ndarray:
        np.random.seed(hash(text) % 2**31)
        return np.random.randn(128).astype(np.float32)
    
    def search(self, query: str, top_k: int = 3) -> List[Dict]:
        query_embed = self._simple_embed(query)
        scores = [np.dot(query_embed, e) / (np.linalg.norm(query_embed) * np.linalg.norm(e) + 1e-8)
                  for e in self.embeddings]
        top_indices = np.argsort(scores)[-top_k:][::-1]
        return [{**self.documents[i], "score": round(scores[i], 4)} for i in top_indices]

# 构建RFFI知识库
rag = SimpleRAG()

knowledge_base = [
    ("rffi_intro", "射频指纹识别(RFFI)通过提取无线设备的硬件特征来识别设备身份。主要指纹来源包括载波频偏(CFO)、IQ不平衡、功率放大器非线性等。"),
    ("cfo_detail", "载波频偏(CFO)由振荡器偏差引起, 不同设备的CFO不同, 是RFFI最重要的指纹特征之一。CFO通常在-500Hz到500Hz范围内。"),
    ("iq_imbalance", "IQ不平衡由混频器缺陷引起, 包括幅度不平衡和相位不平衡。幅度偏差通常在0.9-1.1, 相位偏差在-10到10度。"),
    ("pa_nonlinear", "功率放大器非线性使信号产生谐波和互调失真。三阶截点(IP3)是衡量PA非线性的关键指标。不同设备的PA非线性系数不同。"),
    ("1d_cnn", "1D CNN是RFFI最常用的模型架构, 输入为IQ信号(2,N), 通过多层1D卷积提取时域特征。典型结构: Conv1d->BN->ReLU->MaxPool->FC。"),
    ("open_set", "Open-set识别是RFFI的关键挑战, 需要拒绝未知设备。常用方法包括阈值法、ODIN、度量学习和原型网络。"),
    ("domain_adapt", "域适应解决跨环境泛化问题, 常用方法包括迁移学习、MMD、DANN。源域(实验室)到目标域(实际部署)的性能下降是核心挑战。"),
    ("data_augment", "RF数据增强模拟信道变化, 包括幅度缩放、相位旋转、频偏注入、噪声注入、时间偏移等。数据增强可显著提高模型鲁棒性。"),
]

for doc_id, text in knowledge_base:
    rag.add_document(doc_id, text)

print(f"知识库: {len(rag.documents)} 篇文档")

In [ ]:
# 检索测试
queries = [
    "什么是射频指纹识别?",
    "如何解决跨环境泛化问题?",
    "1D CNN模型怎么设计?",
]

for q in queries:
    results = rag.search(q, top_k=2)
    print(f"查询: {q}")
    for r in results:
        print(f"  [{r['score']:.4f}] {r['text'][:60]}...")
    print()

In [ ]:
# RAG Agent
class RAGAgent(BaseAgent):
    def __init__(self, rag: SimpleRAG, name: str = "RAGAgent"):
        super().__init__(name)
        self.rag = rag
    
    def think(self) -> str:
        return self.memory[-1].content if self.memory else ""
    
    def act(self, query: str) -> str:
        results = self.rag.search(query, top_k=3)
        context = "\n".join([f"[{r['id']}] {r['text']}" for r in results])
        answer = f"基于检索到的知识:\n{context}\n\n回答: 根据以上信息, {query}的相关内容已列出。"
        return f"FINAL ANSWER: {answer}"

rag_agent = RAGAgent(rag)
result = rag_agent.run("如何解决跨环境泛化问题?")
print(result[:300])

---
## 5. 规划与记忆

### 规划 (Planning)

复杂任务需要分解为子任务:

```
任务: 对新采集的IQ数据进行设备认证

分解:
  1. 加载IQ数据
  2. 预处理 (归一化/滤波)
  3. 加载模型
  4. 推理预测
  5. 置信度判断
  6. 返回认证结果
```

### 记忆 (Memory)

| 类型 | 作用 | 实现 |
|------|------|------|
| 短期记忆 | 当前对话上下文 | 消息列表 |
| 长期记忆 | 跨会话知识 | 向量数据库 |
| 工作记忆 | 当前任务状态 | 划掉清单 |
| 情景记忆 | 历史经验 | 案例库 |

In [ ]:
# 规划Agent
class PlanningAgent:
    def __init__(self):
        self.plan: List[Dict] = []
        self.completed: List[str] = []
    
    def create_plan(self, task: str) -> List[Dict]:
        task_plans = {
            "device_auth": [
                {"step": 1, "action": "load_iq", "desc": "加载IQ数据"},
                {"step": 2, "action": "preprocess", "desc": "预处理(归一化/滤波)"},
                {"step": 3, "action": "load_model", "desc": "加载RFFI模型"},
                {"step": 4, "action": "predict", "desc": "推理预测设备ID"},
                {"step": 5, "action": "verify", "desc": "置信度验证"},
                {"step": 6, "action": "report", "desc": "生成认证报告"},
            ],
            "security_audit": [
                {"step": 1, "action": "scan", "desc": "扫描频段发现设备"},
                {"step": 2, "action": "fingerprint", "desc": "提取射频指纹"},
                {"step": 3, "action": "compare", "desc": "与数据库比对"},
                {"step": 4, "action": "detect_anomaly", "desc": "检测异常设备"},
                {"step": 5, "action": "report", "desc": "生成安全报告"},
            ],
        }
        
        for key, plan in task_plans.items():
            if key in task.lower() or any(kw in task.lower() for kw in ["auth", "认证"]):
                self.plan = plan
                return plan
            if key in task.lower() or any(kw in task.lower() for kw in ["audit", "审计", "scan"]):
                self.plan = plan
                return plan
        
        self.plan = task_plans["device_auth"]
        return self.plan
    
    def execute_step(self, step_idx: int) -> str:
        if step_idx >= len(self.plan):
            return "All steps completed"
        step = self.plan[step_idx]
        self.completed.append(step["action"])
        return f"Step {step['step']}: {step['desc']} -> DONE"
    
    def get_status(self) -> str:
        done = len(self.completed)
        total = len(self.plan)
        return f"Progress: {done}/{total} steps completed"

planner = PlanningAgent()
plan = planner.create_plan("device authentication")
print("执行计划:")
for step in plan:
    print(f"  {step}")
print()
for i in range(len(plan)):
    print(planner.execute_step(i))
print(planner.get_status())

In [ ]:
# 记忆系统
class AgentMemory:
    def __init__(self, max_short_term: int = 50):
        self.short_term: List[Dict] = []  # 当前对话
        self.long_term: List[Dict] = []   # 持久化经验
        self.max_short_term = max_short_term
    
    def add_short_term(self, role: str, content: str):
        self.short_term.append({"role": role, "content": content})
        if len(self.short_term) > self.max_short_term:
            # 摘要压缩旧记忆
            old = self.short_term[:10]
            self.short_term = [{"role": "system", "content": f"[Summary of {len(old)} earlier messages]"}] + self.short_term[10:]
    
    def add_long_term(self, key: str, value: str):
        self.long_term.append({"key": key, "value": value})
    
    def get_context(self, query: str = "", max_items: int = 10) -> str:
        recent = self.short_term[-max_items:]
        relevant_long = self.long_term[-3:]  # 简化: 取最近3条
        context_parts = []
        if relevant_long:
            context_parts.append("Long-term memory:")
            for item in relevant_long:
                context_parts.append(f"  {item['key']}: {item['value']}")
        context_parts.append("Recent context:")
        for msg in recent:
            context_parts.append(f"  [{msg['role']}] {msg['content'][:60]}")
        return "\n".join(context_parts)

memory = AgentMemory()
memory.add_short_term("user", "查询设备3的信息")
memory.add_short_term("assistant", "设备3: Sensor-C, LoRa, CFO=350Hz")
memory.add_long_term("device_3_info", "Sensor-C, LoRa, CFO=350Hz, Lab-3")
memory.add_long_term("auth_policy", "置信度阈值0.7, 低于阈值拒绝")

print(memory.get_context())

---
## 6. Multi-Agent 系统

多个Agent协作完成复杂任务:

```
用户请求
    |
    v
协调Agent (Orchestrator)
    |           |           |
    v           v           v
扫描Agent   认证Agent   安全Agent
(发现设备)  (识别身份)  (威胁检测)
    |           |           |
    +-----------+-----------+
                |
                v
            综合报告
```

In [ ]:
# Multi-Agent 系统
class SpecialistAgent:
    def __init__(self, name: str, specialty: str):
        self.name = name
       .specialty = specialty
    
    def process(self, task: str) -> Dict:
        np.random.seed(hash(task + self.name) % 2**31)
        if self.specialty == "scan":
            n = np.random.randint(2, 8)
            return {"agent": self.name, "devices_found": n,
                    "frequencies": [2400 + i*5 for i in range(n)],
                    "result": f"发现{n}个设备"}
        elif self.specialty == "auth":
            n = np.random.randint(2, 8)
            authenticated = np.random.randint(1, n)
            return {"agent": self.name, "total": n, "authenticated": authenticated,
                    "rejected": n - authenticated,
                    "result": f"{authenticated}/{n}设备认证通过"}
        elif self.specialty == "security":
            threats = np.random.choice([0, 1, 2], p=[0.6, 0.3, 0.1])
            return {"agent": self.name, "threats": threats,
                    "level": ["safe", "warning", "critical"][threats],
                    "result": f"检测到{threats}个威胁"}
        return {"agent": self.name, "result": "unknown task"}

class OrchestratorAgent:
    def __init__(self):
        self.agents = {
            "scanner": SpecialistAgent("Scanner", "scan"),
            "authenticator": SpecialistAgent("Authenticator", "auth"),
            "security": SpecialistAgent("Security", "security"),
        }
    
    def run(self, task: str) -> Dict:
        results = {}
        for name, agent in self.agents.items():
            results[name] = agent.process(task)
        
        report = self._synthesize(results)
        return {"task": task, "agent_results": results, "report": report}
    
    def _synthesize(self, results: Dict) -> str:
        scan = results["scanner"]["result"]
        auth = results["authenticator"]["result"]
        sec = results["security"]["result"]
        level = results["security"]["level"]
        return f"扫描完成: {scan}; 认证结果: {auth}; 安全等级: {level} ({sec})"

orchestrator = OrchestratorAgent()
result = orchestrator.run("对当前环境进行安全审计")

print("=== Multi-Agent 协作结果 ===")
print(f"任务: {result['task']}")
print()
for name, res in result["agent_results"].items():
    print(f"{name}: {res['result']}")
print(f"\n综合报告: {result['report']}")

---
## 7. Agent 开发框架

### 主流框架对比

| 框架 | 特点 | 适用场景 |
|------|------|----------|
| LangChain | 生态丰富, 组件多 | 通用Agent开发 |
| LlamaIndex | RAG能力强 | 知识库问答 |
| AutoGen | Multi-Agent | 多Agent协作 |
| CrewAI | 角色扮演Agent | 团队协作任务 |
| Semantic Kernel | 微软出品, 企业级 | .NET/Java集成 |
| Dify | 低代码, 可视化 | 快速搭建Agent |
| OpenAI Assistants | 官方API | 简单场景 |

In [ ]:
# LangChain 风格的Agent伪代码
print("=== LangChain 风格 Agent ===")
print()
print("from langchain.agents import create_react_agent")
print("from langchain.tools import Tool")
print("from langchain_openai import ChatOpenAI")
print()
print("# 定义工具")
print("tools = [")
print("    Tool(name='rf_scan', func=rf_scan, description='扫描射频信号'),")
print("    Tool(name='rf_fingerprint', func=rf_fingerprint, description='提取射频指纹'),")
print("    Tool(name='rf_jamming_detect', func=rf_jamming_detect, description='检测干扰'),")
print("]")
print()
print("# 创建Agent")
print("llm = ChatOpenAI(model='gpt-4')")
print("agent = create_react_agent(llm, tools, prompt)")
print()
print("# 运行")
print("result = agent.invoke({'input': '扫描2.4GHz频段并识别设备'})")
print()
print("# 关键组件:")
print("  llm: 大语言模型 (决策大脑)")
print("  tools: 可调用工具 (行动能力)")
print("  prompt: 系统提示词 (行为规范)")
print("  memory: 对话记忆 (上下文)")

In [ ]:
# Agent 设计模式
print("=== Agent 设计模式 ===")
print()
print("1. ReAct (Reasoning + Acting)")
print("   Thought -> Action -> Observation -> Thought -> ...")
print("   最基础的模式, 适合大多数场景")
print()
print("2. Plan-and-Execute")
print("   先规划完整计划, 再逐步执行")
print("   适合复杂多步任务")
print()
print("3. Reflection (反思)")
print("   执行后自我评估, 改进策略")
print("   提高输出质量")
print()
print("4. LATS (Language Agent Tree Search)")
print("   树搜索 + LLM评估, 找最优路径")
print("   适合需要探索多种可能性的任务")
print()
print("5. Multi-Agent")
print("   多个Agent分工协作")
print("   适合复杂系统")
print()
print("6. Toolformer")
print("   模型自主学习使用工具")
print("   减少人工设计")

In [ ]:
# RFFI Agent 完整设计
print("=== RFFI Agent 完整设计 ===")
print()
print("系统架构:")
print()
print("  用户/Gateway")
print("       |")
print("       v")
print("  Orchestrator Agent")
print("       |")
print("       +-- Scanner Agent")
print("       |     工具: rf_scan, spectrum_analyze")
print("       |     职责: 发现设备, 监测频谱")
print("       |")
print("       +-- Auth Agent")
print("       |     工具: rf_fingerprint, model_predict")
print("       |     职责: 提取指纹, 设备认证")
print("       |")
print("       +-- Security Agent")
print("       |     工具: jamming_detect, anomaly_detect")
print("       |     职责: 威胁检测, 异常告警")
print("       |")
print("       +-- Knowledge Agent (RAG)")
print("             知识库: RFFI论文, 设备数据库, 历史记录")
print("             职责: 知识检索, 经验查询")
print()
print("工作流程:")
print("  1. Scanner 发现新设备")
print("  2. Auth 提取指纹并认证")
print("  3. Security 检测潜在威胁")
print("  4. Knowledge 提供历史参考")
print("  5. Orchestrator 综合决策")

In [ ]:
# Agent 开发最佳实践
print("=== Agent 开发最佳实践 ===")
print()
print("1. 提示词工程 (Prompt Engineering)")
print("   - 明确角色: '你是一个射频安全专家'")
print("   - 约束行为: '只能使用提供的工具'")
print("   - 输出格式: '以JSON格式返回结果'")
print()
print("2. 工具设计")
print("   - 描述清晰: LLM根据描述选择工具")
print("   - 参数简洁: 减少必填参数")
print("   - 错误处理: 工具失败时返回有用信息")
print()
print("3. 安全性")
print("   - 限制工具权限: 不能执行危险操作")
print("   - 输入校验: 防止注入攻击")
print("   - 人工确认: 关键操作需人工审批")
print()
print("4. 可靠性")
print("   - 重试机制: 工具调用失败时重试")
print("   - 超时控制: 防止无限循环")
print("   - 回退策略: 主方案失败时用备选方案")
print()
print("5. 评估")
print("   - 任务完成率")
print("   - 工具调用准确率")
print("   - 平均步数")
print("   - 延迟")

---
## 总结

### Agent 核心概念

```
Agent = LLM + Tools + Memory + Planning

LLM:     决策大脑 (选择工具、推理、生成)
Tools:   行动能力 (API、数据库、模型推理)
Memory:  上下文记忆 (短期+长期)
Planning: 任务规划 (分解、排序、执行)
```

### Agent 设计模式

| 模式 | 核心思想 | 适用场景 |
|------|---------|----------|
| ReAct | 思考-行动-观察循环 | 通用 |
| Plan-Execute | 先规划再执行 | 复杂多步 |
| RAG | 检索增强生成 | 知识密集型 |
| Multi-Agent | 多Agent协作 | 大型系统 |
| Reflection | 自我反思改进 | 质量要求高 |

### RFFI Agent 架构

```
Orchestrator
  +-- Scanner Agent (发现设备)
  +-- Auth Agent (指纹认证)
  +-- Security Agent (威胁检测)
  +-- Knowledge Agent (知识检索)
```

### 关键技术栈

| 组件 | 推荐技术 |
|------|----------|
| LLM | GPT-4 / Claude / 开源模型 |
| 框架 | LangChain / LlamaIndex |
| 向量库 | Chroma / FAISS / Milvus |
| 部署 | FastAPI + Docker |
| 监控 | LangSmith / Phoenix |